# BƯỚC 2: TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING)

**Mục tiêu:** Làm sạch, ghép nối và chuẩn hóa dữ liệu thô đã thu thập được ở Bước 1 để chuẩn bị đưa vào mô hình Học máy.

**Các bước thực hiện chuẩn barem:**
1. **Nạp dữ liệu:** Đọc các file `.csv` từ thư mục `../data/raw/`.
2. **Ghép nối dữ liệu (Data Merging):** Tích hợp chỉ số thị trường (VNINDEX) và tỷ giá (USD/VND) vào từng mã cổ phiếu để làm giàu đặc trưng vĩ mô.
3. **Xử lý khuyết thiếu (Missing Values):** Tuyệt đối không dùng `dropna()` gây mất thông tin chuỗi thời gian. Sử dụng kỹ thuật `ffill` và `bfill`.
4. **Chuẩn hóa (Feature Scaling):** Dùng `MinMaxScaler` và bắt buộc chỉ `fit` trên tập Train (đến hết năm 2024) để chống rò rỉ dữ liệu (Data Leakage).

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import os
import pickle
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

print("Loading raw data from '../data/raw/'...")

data_frames = {}
raw_files = ['VNM_raw.csv', 'FPT_raw.csv', 'VIC_raw.csv', 'VNINDEX_raw.csv', 'USD_VND_raw.csv']
data_dir = "../data/raw"

for file in raw_files:
    file_path = os.path.join(data_dir, file)
    
    #Kiểm tra file không tồn tại thì skip
    if not os.path.exists(file_path):
        print(f"[CẢNH BÁO] Không tìm thấy file: {file}")
        continue
        
    name = file.replace('_raw.csv', '')
    df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
    data_frames[name] = df
    
    print(f"-> Đã tải {name} ({len(df)} dòng)")

## 2.1. Ghép nối dữ liệu (Data Merging)

Để AI có được "góc nhìn vĩ mô" toàn cảnh, ta sẽ lấy dữ liệu của từng mã cổ phiếu (VNM, FPT, VIC) làm gốc, sau đó ghép nối (Left Join) thêm các cột của VNINDEX và tỷ giá USD/VND theo đúng trục thời gian (`Date`).

In [ ]:
print("--- 2.1: Ghép nối dữ liệu (Merge Data) ---")

merged_data = {}
target_tickers = ['VNM', 'FPT', 'VIC']

#Chuẩn bị data VNINDEX
df_vn = data_frames.get('VNINDEX')
if df_vn is not None:
    df_vn = df_vn.add_prefix('VNINDEX_')

#Chuẩn bị data USD/VND
df_usd = data_frames.get('USD_VND')
if df_usd is not None and 'Close' in df_usd.columns:
    df_usd = df_usd[['Close']].rename(columns={'Close': 'USDVND_Close'})

#Tiến hành Left Join
for ticker in target_tickers:
    if ticker not in data_frames:
        print(f"[{ticker}] Thiếu dữ liệu gốc, bỏ qua merge.")
        continue
        
    df_stock = data_frames[ticker].copy()
    
    
    if df_vn is not None:
        df_stock = df_stock.join(df_vn, how='left')
        
    if df_usd is not None:
        df_stock = df_stock.join(df_usd, how='left')
        
    merged_data[ticker] = df_stock
    print(f"-> Đã merge {ticker} ({df_stock.shape[1]} cột)")

## 2.2. Xử lý giá trị khuyết thiếu (Missing Values)

Thị trường chứng khoán không giao dịch vào Thứ 7, Chủ Nhật và ngày Lễ, dẫn đến việc dữ liệu bị đứt đoạn. Việc dùng `dropna()` sẽ làm hỏng tính liên tục của chuỗi thời gian (Time Series). 
**Giải pháp:** - Tạo một trục thời gian chuẩn 365 ngày/năm.
- Dùng `ffill()` (Forward Fill) để lấy giá đóng cửa của ngày Thứ 6 điền cho Thứ 7 và Chủ nhật.
- Dùng `bfill()` (Backward Fill) để xử lý nốt các ô trống ở đầu chuỗi (nếu có).

In [ ]:
print("--- 2.2: Xử lý Missing Values (ffill & bfill) ---")
start_date = '2020-01-01'
end_date = pd.Timestamp.today().normalize()
full_date_range = pd.date_range(start=start_date, end=end_date, freq='D')

filled_data = {}

for ticker, df in merged_data.items():
    #Xóa timezone và loại bỏ các dòng bị trùng ngày
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df[~df.index.duplicated(keep='last')]
    
    #Ép khung thời gian liên tục
    df_reindexed = df.reindex(full_date_range)
    df_reindexed.index.name = 'Date'
    
    nan_before = df_reindexed.isna().sum().sum()
    
    #Lấp đầy dữ liệu: ffill (kéo giá cũ tới) rồi bfill
    df_filled = df_reindexed.ffill().bfill()
    nan_after = df_filled.isna().sum().sum()
    
    filled_data[ticker] = df_filled
    
    print(f"-> {ticker}: Đã fill {nan_before} NaNs. CÒn lại: {nan_after}")

## 2.3. Chuẩn hóa dữ liệu (Feature Scaling) và Chống Rò rỉ (Data Leakage)

Các biến số có thang đo rất khác nhau (Ví dụ: Giá VIC là 40.000 VND, trong khi tỷ giá USD là 25.000, Volume có thể lên tới hàng triệu). Ta cần dùng `MinMaxScaler` để đưa tất cả về khoảng `[0, 1]`.


In [ ]:
print("--- 2.3: Chuẩn hóa dữ liệu (Scaling) & Lưu trữ ---")
split_date = '2024-12-31'
model_dir = '../models'
processed_dir = '../data/processed'

os.makedirs(model_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

for ticker, df in filled_data.items():
    df_scaled = df.copy()
    cols = df_scaled.columns 
    
    #Khởi tạo và Fit Scaler CHỈ trên tập Train
    train_mask = df_scaled.index <= split_date
    
    scaler = MinMaxScaler()
    scaler.fit(df_scaled.loc[train_mask, cols])
    
    #Transform toàn bộ dữ liệu
    df_scaled[cols] = scaler.transform(df_scaled[cols])
    
    #Lưu Scaler model
    scaler_path = os.path.join(model_dir, f"scaler_{ticker}.pkl")
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
        
    #Lưu dữ liệu đã chuẩn hóa
    csv_path = os.path.join(processed_dir, f"{ticker}_scaled.csv")
    df_scaled.to_csv(csv_path)
    
    print(f"-> Hoàn tất {ticker}: Đã xuất file .csv và model .pkl")